# Exercises: Hypothesis Testing

## Name: Shaima Alqahtani

**A Waiter's Tips**

The following description was retrieved from Kaggle page.

> Food servers’ tips in restaurants may be influenced by many factors, including the nature of the restaurant, size of the party, and table locations in the restaurant. Restaurant managers need to know which factors matter when they assign tables to food servers. For the sake of staff morale, they usually want to avoid either the substance or the appearance of unfair treatment of the servers, for whom tips (at least in restaurants in the United States) are a major component of pay. In one restaurant, a food server recorded the following data on all customers they served during an interval of two and a half months in early 1990. The restaurant, located in a suburban shopping mall, was part of a national chain and served a varied menu. In observance of local law, the restaurant offered to seat in a non-smoking section to patrons who requested it. Each record includes a day and time, and taken together, they show the server’s work schedule.

Acknowledgements The data was reported in a collection of case studies for business statistics. Bryant, P. G. and Smith, M (1995) Practical Data Analysis: Case Studies in Business Statistics. Homewood, IL: Richard D. Irwin Publishing

The dataset is also available through the Python package Seaborn.

In [13]:
import seaborn as sns

tips = sns.load_dataset("tips")

Here's a description of each column in the dataset:

- `total_bill`: The total bill amount, including the cost of food and drinks.
- `tip`: The tip amount given by the customer.
- `sex`: The gender of the customer (e.g., Male or Female).
- `smoker`: Whether the customer is a smoker or not (e.g., Yes or No).
- `day`: The day of the week when the transaction occurred (e.g., Sun, Sat, Thu, etc.).
- `time`: The time of day when the transaction occurred, typically categorized as Lunch or Dinner.
- `size`: The size of the party or group of customers.

**Your Task**: is to accept or reject the following hypothesis using statistical testing:

- Hypothesis $H_1$: smoking is associated with time of visit
- Hypothesis $H_2$: the bigger the group the higher the tip
- Hypothesis $H_3$: group size is different based on the time of visit
- Hypothesis $H_4$: (... come up with a hypothesis of your own ...)
- Finally, analyze if size (party size) is a **confounder**. That is, does a larger party cause a higher tip, or simply a higher bill which then leads to a higher tip?

---

In [14]:
import seaborn as sns
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

tips = sns.load_dataset("tips")
tips

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [15]:
# H1:
ct_smoker_time = pd.crosstab(tips["smoker"], tips["time"])

chi2, p, dof, expected = stats.chi2_contingency(ct_smoker_time)

print("Chi-square:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:\n", expected)

Chi-square: 0.5053733928754355
p-value: 0.4771485672079724
Degrees of freedom: 1
Expected frequencies:
 [[ 25.91803279  67.08196721]
 [ 42.08196721 108.91803279]]


In [16]:
# H2
pearson_r, pearson_p = stats.pearsonr(tips["size"], tips["tip"])
spearman_r, spearman_p = stats.spearmanr(tips["size"], tips["tip"])

print("Pearson r:", pearson_r, "p-value:", pearson_p)
print("Spearman r:", spearman_r, "p-value:", spearman_p)


Pearson r: 0.48929877523035725 p-value: 4.3005433272249666e-16
Spearman r: 0.46826792926211475 p-value: 1.059850307726079e-14


In [17]:
# H3
size_lunch = tips.loc[tips["time"] == "Lunch", "size"]
size_dinner = tips.loc[tips["time"] == "Dinner", "size"]

t_stat, p_val = stats.ttest_ind(size_lunch, size_dinner, equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_val)

t-statistic: -1.5247411188532956
p-value: 0.130223295108278


In [18]:
# H4
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

tip_pct_smoker = tips.loc[tips["smoker"] == "Yes", "tip_pct"]
tip_pct_nonsmoker = tips.loc[tips["smoker"] == "No", "tip_pct"]

t_stat4, p_val4 = stats.ttest_ind(tip_pct_smoker, tip_pct_nonsmoker, equal_var=False)

print("H4 - t-statistic:", t_stat4)
print("H4 - p-value:", p_val4)


H4 - t-statistic: 0.4112251378479512
H4 - p-value: 0.681657919088972


In [19]:
# confounder
model1 = smf.ols("tip ~ size", data=tips).fit()
print(model1.summary())

model2 = smf.ols("tip ~ size + total_bill", data=tips).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.239
Model:                            OLS   Adj. R-squared:                  0.236
Method:                 Least Squares   F-statistic:                     76.18
Date:                Sat, 11 Apr 2026   Prob (F-statistic):           4.30e-16
Time:                        23:00:46   Log-Likelihood:                -391.56
No. Observations:                 244   AIC:                             787.1
Df Residuals:                     242   BIC:                             794.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.1691      0.223      5.233      0.0

In [20]:
print("corr(size, tip):", tips["size"].corr(tips["tip"]))
print("corr(size, total_bill):", tips["size"].corr(tips["total_bill"]))
print("corr(total_bill, tip):", tips["total_bill"].corr(tips["tip"]))

corr(size, tip): 0.48929877523035775
corr(size, total_bill): 0.5983151309049022
corr(total_bill, tip): 0.6757341092113641
